<div style="background-color: black; color: white; padding: 10px;text-align: center;">
  <strong>Date Published:</strong> Aug 30, 2026 <strong>Author:</strong> Adnan Alaref
</div>

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 1: Import Library.</div>

In [24]:
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

import warnings
warnings.simplefilter(action= "ignore")
warnings.filterwarnings(action= "ignore", category=FutureWarning)

In [25]:
def set_all_seeds(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 2: Fixed-Attention Mechanism.</div>

### **Equation:**
$$
\text{Output}
=
\text{softmax}
\left(
\frac{MM^T}{\sqrt{d}}
\right)M
$$

---

```
             word1 word2 word3 ... word100
word1          ✓     ✓     ✓
word2          ✓     ✓     ✓
word3          ✓     ✓     ✓
...
word100        ✓     ✓     ✓

```

---

```
        word1  word2  word3  word4
word1   2.1    0.5    1.2    3.0

We want Softmax to normalize across this row:

        word1  word2  word3  word4
word1   0.20   0.04   0.08   0.68
                ↑
              sum = 1
```

In [26]:
# ── Set seed for reproducibility ──────────────────────────────────────────────
set_all_seeds(42)

# ── Fixed Self-Attention ──────────────────────────────────────────────────────
class FixedAttention(nn.Module):
  def __init__(self, embedding_dim: int) -> None:
    super().__init__()
    self.scale = embedding_dim ** 0.5

  def forward(self, M: torch.Tensor)-> torch.Tensor:
    # [N, D] @ [D, N] → [N, N] --> 100x13 * 13*100 = 100x100
    scores = torch.matmul(M, M.T) / self.scale

    # Normalize each row vector
    scores = F.softmax(scores, dim = -1)

    # [N, N] @ [N, D] → [N, D]
    output = torch.matmul(scores, M)  # 100x13

    return output

In [27]:
# ── Test Fixed Attention ──────────────────────────────────────────────────────
words = 100                   # Number of words
embed_dim = 13                # Embedding dimension

data = nn.Embedding(words, embed_dim).weight  # [100, 13]
# nn.Parameter ⊂ torch.Tensor
print(type(data))

fixed_attention = FixedAttention(embed_dim)

reweighted_data = fixed_attention(data)       # [100, 13]

print(reweighted_data.shape)                  # Check output shape
print(data)                                   # Original embeddings
print(reweighted_data)                        # Contextualized embeddings

<class 'torch.nn.parameter.Parameter'>
torch.Size([100, 13])
Parameter containing:
tensor([[ 1.9269,  1.4873,  0.9007,  ..., -0.3925, -1.4036, -0.7279],
        [-0.5594, -0.7688,  0.7624,  ...,  1.6806,  1.2791,  1.2964],
        [ 0.6105,  1.3347, -0.2316,  ...,  0.3189, -0.4245,  0.3057],
        ...,
        [-0.5622,  0.8253,  2.2683,  ..., -0.7519,  2.4401, -1.9129],
        [ 0.3108, -1.4763, -0.4783,  ..., -0.7610, -1.2726, -0.6267],
        [ 1.3713, -0.4387, -1.3193,  ...,  1.1428, -0.5947,  0.5363]],
       requires_grad=True)
tensor([[ 1.3864,  1.0576,  0.7042,  ..., -0.3054, -1.0667, -0.4332],
        [-0.1454, -0.3221,  0.3081,  ...,  0.7183,  0.4251,  0.5036],
        [ 0.2007,  0.4715,  0.0120,  ...,  0.1296, -0.1775,  0.1401],
        ...,
        [-0.4484,  0.6942,  1.8028,  ..., -0.6447,  2.0627, -1.5922],
        [-0.1322, -0.5115, -0.2304,  ..., -0.2959, -0.3701, -0.2922],
        [ 0.3928, -0.1483, -0.4308,  ...,  0.4504, -0.1128,  0.1358]],
       grad_fn=<MmBack

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 3: Self-Attention Mechanism.</div>

**The self-attention mechanism is defined as:**
### **Equation:**

$$
\text{Attention}(Q,K,V)
=
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

**Where:**

- $Q = XW_Q$ → **Query matrix**
- $K = XW_K$ → **Key matrix**
- $V = XW_V$ → **Value matrix**
- $X$ → Input embeddings
- $W_Q$ → Learnable **Query projection** matrix
- $W_K$ → Learnable **Key projection** matrix
- $W_V$ → Learnable **Value projection** matrix
- $d_k$ → Dimension of the Key vectors
- $QK^T$ → Similarity scores between queries and keys
- $\frac{QK^T}{\sqrt{d_k}}$ → Scaled attention scores
- $\text{softmax}(\cdot)$ → Converts scores into attention weights
- $V$ → Information that is combined using the attention weights
- $\text{Attention}(Q,K,V)$ → Contextualized output representations

In [28]:
# ── Set seed for reproducibility ──────────────────────────────────────────────
set_all_seeds(42)

# ── Self-Attention ────────────────────────────────────────────────────────────
class SelfAttention(nn.Module):
  def __init__(self, embedding_dim: int ) -> None:
    super().__init__()

    self.scale = embedding_dim ** 0.5

    self.query = nn.Linear(embedding_dim, embedding_dim)
    self.key = nn.Linear(embedding_dim, embedding_dim)
    self.value = nn.Linear(embedding_dim, embedding_dim)


  def forward(self, M: torch.Tensor)-> torch.Tensor:
    key = self.key(M)
    query = self.query(M)
    value = self.value(M)

    scores = torch.matmul(query, key.T) / self.scale
    scores = F.softmax(scores, dim=-1)

    output = torch.matmul(scores, value)

    return output

In [29]:
# ── Test Self-Attention ───────────────────────────────────────────────────────
words = 100                   # Number of words
embed_dim = 13                # Embedding dimension

data = nn.Embedding(words, embed_dim).weight  # [100, 13]
self_attention = SelfAttention(embed_dim)
reweighted_data = self_attention(data)

print(reweighted_data.shape)                  # Check output shape
print(data)                                   # Original embeddings
print(reweighted_data)                        # Contextualized embeddings

torch.Size([100, 13])
Parameter containing:
tensor([[ 1.9269,  1.4873,  0.9007,  ..., -0.3925, -1.4036, -0.7279],
        [-0.5594, -0.7688,  0.7624,  ...,  1.6806,  1.2791,  1.2964],
        [ 0.6105,  1.3347, -0.2316,  ...,  0.3189, -0.4245,  0.3057],
        ...,
        [-0.5622,  0.8253,  2.2683,  ..., -0.7519,  2.4401, -1.9129],
        [ 0.3108, -1.4763, -0.4783,  ..., -0.7610, -1.2726, -0.6267],
        [ 1.3713, -0.4387, -1.3193,  ...,  1.1428, -0.5947,  0.5363]],
       requires_grad=True)
tensor([[-0.2497,  0.1938, -0.2641,  ...,  0.3098,  0.0961, -0.3424],
        [-0.1707,  0.2021, -0.1268,  ...,  0.2980,  0.0907, -0.0588],
        [-0.2772,  0.2213, -0.2436,  ...,  0.2545,  0.1106, -0.2408],
        ...,
        [-0.3270,  0.1655, -0.2436,  ...,  0.1731,  0.1365, -0.1254],
        [-0.2884,  0.2168, -0.2391,  ...,  0.2562,  0.1101, -0.2472],
        [-0.2277,  0.2229, -0.2264,  ...,  0.2334,  0.1165, -0.1508]],
       grad_fn=<MmBackward0>)


# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 4: Multi-Head Self-Attention Mechanism.</div>

**Multi-Head Self-Attention applies self-attention in multiple parallel heads.**
- The embedding dimension **$D$** is **divided** across the **$h$** attention heads.
- For example, if $D=512$ and $h=8$, each head gets $d_k=d_v=\frac{512}{8}=64$ dimensions.
---
### **Equation:**
$$
\text{head}_i
=
\text{softmax}
\left(
\frac{Q_iK_i^T}{\sqrt{d_k}}
\right)V_i
$$

**where:**

$$
Q_i = XW_i^Q,\qquad
K_i = XW_i^K,\qquad
V_i = XW_i^V
$$

<h3 align="center">The outputs of all heads are concatenated and projected</h3>

$$
\boxed{
\text{MultiHead}(X)
=
\text{Concat}(\text{head}_1,\ldots,\text{head}_h)W^O
}
$$

### **Parameters**
- $X$ → Input sequence / embeddings
- $h$ → Number of attention heads
- $Q_i$ → Query matrix for head $i$
- $K_i$ → Key matrix for head $i$
- $V_i$ → Value matrix for head $i$
- $W_i^Q$ → Learnable Query projection matrix for head $i$
- $W_i^K$ → Learnable Key projection matrix for head $i$
- $W_i^V$ → Learnable Value projection matrix for head $i$
- $d_k$ → Dimension of the Key vectors in each head
- $\text{head}_i$ → Output of attention head $i$
- $\text{Concat}$ → Concatenates the outputs of all heads
- $W^O$ → Learnable output projection matrix
- $\text{MultiHead}(X)$ → Final multi-head attention output

---
### **Multi-Head Self-Attention Flow**
>**Input Embedding** → **Split into Heads** → **Compute Attention for Each Head** → **Concatenate Heads** → **Output Projection**

---
### **Note:**
> `transpose(a, b)` and `permute(a, b, c, ...)` can make the tensor non-contiguous in memory.  
> Therefore, `.contiguous()` is required before `.view()`.  
> `.reshape()` can handle non-contiguous tensors, so `.contiguous()` is not required.ired.

In [30]:
# ── Set seed for reproducibility ──────────────────────────────────────────────
set_all_seeds(42)

# ── Multi-Head Self-Attention ─────────────────────────────────────────────────
class MultiHeadSelfAttention(nn.Module):
  def __init__(self, heads: int, embedding_dim: int) -> None:
    super().__init__()

    self.heads = heads
    self.head_dim = embedding_dim // heads

    assert self.head_dim * heads == embedding_dim,\
    f"embedding_dim ({embedding_dim}) must be divisible by heads ({heads})"

    # Scale by the dimension of each head
    self.scale = self.head_dim ** 0.5

    self.queries = nn.Linear(embedding_dim, embedding_dim)
    self.keys = nn.Linear(embedding_dim, embedding_dim)
    self.values = nn.Linear(embedding_dim, embedding_dim)

    self.fc_proj = nn.Linear(embedding_dim, embedding_dim)

  def forward(self, M: torch.Tensor)->torch.Tensor:
    B, seq_len, _ = M.shape

    # ex: [B, seq_len, 512] → [B, seq_len, 8, 64] → [B, 8, seq_len, 64]
    keys = self.keys(M).view(B, seq_len, self.heads, self.head_dim).transpose(1, 2)
    values = self.values(M).view(B, seq_len, self.heads, self.head_dim).transpose(1, 2)
    queries = self.queries(M).view(B, seq_len, self.heads, self.head_dim).transpose(1, 2)

    # [B, H, N, D/H] @ [B, H, D/H, N] → [B, H, N, N]
    # ex: [32, 8, 100, 64] @ [32, 8, 64, 100] = [32, 8, 100, 100]
    scores = torch.matmul(queries, keys.transpose(-2, -1)) / self.scale

    # Normalize attention scores
    scores = F.softmax(scores, dim=-1)

    # ex: [32, 8, 100, 100] @ [32, 8, 100, 64] = [32, 8, 100, 64]
    output = torch.matmul(scores, values)

    # Reshape and combine heads
    # ex: [32, 8, 100, 64] → [32, 100, 8, 64] → [32, 100, 512]
    output = output.transpose(1, 2).contiguous().view(B, seq_len, self.heads * self.head_dim)
    # output = output.transpose(1, 2).reshape(B, seq_len, self.heads * self.head_dim)

    # Final linear projection
    context = self.fc_proj(output)

    return context

In [31]:
# ── Test Multi-Head Self-Attention ────────────────────────────────────────────

# 100 words, each represented by a 512-dimensional embedding
words = 100
embedding_dim = 512

# Number of attention heads
heads = 8

# Create input embeddings: [100, 512]
data = nn.Embedding(words, embedding_dim).weight

# Add batch dimension: [1, 100, 512]
data = data.unsqueeze(0)

# Initialize Multi-Head Self-Attention
attention = MultiHeadSelfAttention(heads, embedding_dim)

# Apply self-attention
output = attention(data)

# Check input and output shapes
print(f"Input shape:  {data.shape}")
print(f"Output shape: {output.shape}")

Input shape:  torch.Size([1, 100, 512])
Output shape: torch.Size([1, 100, 512])


In [32]:
# ── Test Multi-Head Self-Attention with a more realistic batch size of 32 ─────

B = 32                 # Batch size
seq_len = 100          # Number of tokens
embedding_dim = 512    # Embedding dimension
heads = 8              # Number of attention heads

# Each head receives 512 / 8 = 64 dimensions
data = torch.randn(B, seq_len, embedding_dim)
attention = MultiHeadSelfAttention(heads=heads, embedding_dim= embedding_dim)

output = attention(data)

print(f"Input shape:  {data.shape}")
print(f"Output shape: {output.shape}")

Input shape:  torch.Size([32, 100, 512])
Output shape: torch.Size([32, 100, 512])


# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 5: Cross-Attention Mechanism.</div>

In Cross-Attention, **Queries come from one sequence**, while **Keys and Values come from another sequence**.

### **Equation:**
$$
\text{CrossAttention}(Q,K,V)
=
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

**where:**

- $Q$ → Query matrix from the **target/decoder sequence**
- $K$ → Key matrix from the **source/encoder sequence**
- $V$ → Value matrix from the **source/encoder sequence**
- $d_k$ → Dimension of each Key vector
- $QK^T$ → Similarity between target queries and source keys
- $\text{softmax}(\cdot)$ → Converts similarity scores into attention weights
- $V$ → Source information that is combined using the attention weights
- $\text{CrossAttention}(Q,K,V)$ → Contextualized representation of the target sequence

---
### **Cross-Attention Example: English → Arabic**
```text
English: "I love cats"   → Encoder → K,V
                                      ↓
                                 Cross-Attention
                                      ↑
Arabic: "أنا أحب القطط"  ← Decoder ←  Q
```
---

| Arabic token | Attends mostly to |
|---|---|
| أنا | I |
| أحب | love |
| القطط | cats |

In [33]:
# ── Set seed for reproducibility ──────────────────────────────────────────────
set_all_seeds(42)

# ── Cross-Attention ──────────────────────────────────────────────────────────
class CrossAttention(nn.Module):
  def __init__(self, embed_dim: int)->None:
    super().__init__()

    self.scale = embed_dim ** 0.5

    self.query = nn.Linear(embed_dim, embed_dim)
    self.key = nn.Linear(embed_dim, embed_dim)
    self.value = nn.Linear(embed_dim, embed_dim)

  def forward(self, source: torch.Tensor, target: torch.Tensor)-> torch.Tensor:
    # Q comes from the target/output sequence
    query = self.query(target)

    # K and V come from the source/input sequence
    key = self.key(source)
    value = self.value(source)

    # [B, N_target, D] @ [B, D, N_source] → [B, N_target, N_source]
    scores = torch.matmul(query, key.transpose(-2, -1)) / self.scale

    # Normalize across the source tokens
    scores = F.softmax(scores, dim=-1)

    # [B, N_target, N_source] @ [B, N_source, D] → [B, N_target, D]
    output = torch.matmul(scores, value)

    return output

In [34]:
# ── Test Cross-Attention: English → Arabic ────────────────────────────────────

B = 32                 # Batch size
english_len = 100      # Number of English tokens
arabic_len = 20        # Number of Arabic tokens
embed_dim = 512        # Embedding dimension

# English encoder output → K and V
english = torch.randn(B, english_len, embed_dim)

# Arabic decoder ouput → Q
arabic = torch.randn(B, arabic_len, embed_dim)

# Initialize Cross-Attention
cross_attention = CrossAttention(embed_dim)

# Arabic queries attend to English keys and values
context = cross_attention(english, arabic)

# Check shapes
print(f"English shape: {english.shape}")
print(f"Arabic shape:  {arabic.shape}")
print(f"Context shape: {context.shape}")

English shape: torch.Size([32, 100, 512])
Arabic shape:  torch.Size([32, 20, 512])
Context shape: torch.Size([32, 20, 512])


# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Thanks & Upvote ❤️</div>